# ADNET — Duplicate / Near-Duplicate Leakage Check (Kaggle version)

This notebook answers Reviewer 1's Point 2 and Reviewer 2's Point 4: whether any MRI slice appears more than once (or in a near-identical form) across your dataset, and whether any such duplicates cross a train/val/test split boundary.

**Setup steps on Kaggle (do these before running):**
1. Create a new notebook at https://www.kaggle.com/code → **New Notebook**
2. Upload this file via **File → Import Notebook**, or copy these cells in manually
3. On the right-hand panel, click **Add Input** → search for **"Alzheimer's Dataset (4 class of Images)"** (by tourist55) → click **Add**. This is the same public dataset cited in the manuscript — no download needed, it mounts directly into the notebook.
4. Click **Run All** at the top (Settings → GPU is not required for this notebook; it only reads image files, no model training)
5. When it finishes, scroll to the bottom output cell, copy the block starting with the row of `=` characters, and send it back

If you'd rather not use Kaggle's UI dataset search, the fallback cell below shows how to reference a dataset you've added under a different name — just adjust the path.

## 1. Install the one extra package needed (everything else is preinstalled on Kaggle)

In [ ]:
!pip install imagehash --quiet
print('Done.')

## 2. Locate the dataset under /kaggle/input

Kaggle mounts every dataset you added under `/kaggle/input/<dataset-slug>/`. This cell auto-detects it so you don't need to know the exact folder name.

In [ ]:
import os

print('Contents of /kaggle/input:')
for entry in os.listdir('/kaggle/input'):
    print(' -', entry)

if not os.listdir('/kaggle/input'):
    raise RuntimeError(
        "No dataset found under /kaggle/input. Click 'Add Input' in the right-hand "
        "panel and add the 'Alzheimer's Dataset (4 class of Images)' dataset, then "
        "re-run this cell."
    )

# Use the first (and normally only) dataset found. If you've added more than one
# dataset to this notebook, set dataset_path manually instead, e.g.:
# dataset_path = '/kaggle/input/alzheimers-dataset-4-class-of-images'
dataset_path = os.path.join('/kaggle/input', os.listdir('/kaggle/input')[0])
print('\nUsing dataset path:', dataset_path)

for root, dirs, files in os.walk(dataset_path):
    level = root.replace(dataset_path, '').count(os.sep)
    imgs = [f for f in files if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    print('  ' * level + os.path.basename(root) + '/', f'({len(imgs)} images)' if imgs else '')

## 3. Create a fresh stratified 70/15/15 split (matching the manuscript's stated protocol)

This mirrors Section 5.1's description of the split. If you have your own exact split index files from the original experiments, skip this cell and see the note at the bottom instead.

Kaggle notebooks have a read-only `/kaggle/input` and a writable `/kaggle/working` — the split copies files into `/kaggle/working/split_dataset`.

In [ ]:
import glob, random, shutil
random.seed(42)

candidate_dirs = []
for root, dirs, files in os.walk(dataset_path):
    img_files = [f for f in files if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    if img_files:
        candidate_dirs.append((root, len(img_files)))

print('Folders containing images:')
for d, n in candidate_dirs:
    print(f'  {d}  ({n} images)')

split_root = '/kaggle/working/split_dataset'
for split in ['train', 'val', 'test']:
    os.makedirs(os.path.join(split_root, split), exist_ok=True)

for class_dir, n in candidate_dirs:
    class_name = os.path.basename(class_dir)
    imgs = glob.glob(os.path.join(class_dir, '*'))
    imgs = [f for f in imgs if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    random.shuffle(imgs)
    n_train = int(0.70 * len(imgs))
    n_val = int(0.15 * len(imgs))
    splits = {
        'train': imgs[:n_train],
        'val': imgs[n_train:n_train + n_val],
        'test': imgs[n_train + n_val:],
    }
    for split, files in splits.items():
        out_dir = os.path.join(split_root, split, class_name)
        os.makedirs(out_dir, exist_ok=True)
        for f in files:
            # Symlink instead of copy -- Kaggle working-directory disk quota is
            # limited (20GB), and this dataset can be copied twice (train+val+test)
            # which is wasteful; symlinks avoid duplicating the actual image bytes.
            link_path = os.path.join(out_dir, os.path.basename(f))
            if not os.path.exists(link_path):
                os.symlink(f, link_path)

for split in ['train', 'val', 'test']:
    total = sum(len(files) for _, _, files in os.walk(os.path.join(split_root, split)))
    print(f'{split}: {total} images')

## 4. Run the leakage check

Same script referenced in the manuscript and response letter (`duplicate_leakage_check.py`), included here inline.

In [ ]:
import itertools
from pathlib import Path
import imagehash
from PIL import Image
import pandas as pd


def collect_images(directory, split_name):
    exts = {'.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff'}
    records = []
    for root, _, files in os.walk(directory):
        for f in files:
            if Path(f).suffix.lower() in exts:
                cls = Path(root).name
                records.append({'path': os.path.join(root, f), 'split': split_name, 'class': cls})
    return records


def compute_hashes(records, hash_size):
    for r in records:
        try:
            with Image.open(r['path']) as img:
                r['phash'] = imagehash.phash(img, hash_size=hash_size)
        except Exception as e:
            r['phash'] = None
            r['error'] = str(e)
    return records


def find_duplicates(records, near_dup_threshold):
    exact_groups = {}
    for r in records:
        if r['phash'] is None:
            continue
        exact_groups.setdefault(str(r['phash']), []).append(r)
    exact_dupes = {k: v for k, v in exact_groups.items() if len(v) > 1}

    near_dupe_pairs = []
    items = [r for r in records if r['phash'] is not None]
    for a, b in itertools.combinations(items, 2):
        dist = a['phash'] - b['phash']
        if 0 < dist <= near_dup_threshold:
            near_dupe_pairs.append((a, b, dist))
    return exact_dupes, near_dupe_pairs


def summarize(exact_dupes, near_dupe_pairs, out_csv):
    rows = []
    cross_split_exact = 0
    for h, group in exact_dupes.items():
        splits = {g['split'] for g in group}
        crosses = len(splits) > 1
        cross_split_exact += int(crosses)
        for g in group:
            rows.append({'type': 'exact', 'hash': h, 'path': g['path'], 'split': g['split'], 'class': g['class'], 'crosses_split': crosses})

    cross_split_near = 0
    for a, b, dist in near_dupe_pairs:
        crosses = a['split'] != b['split']
        cross_split_near += int(crosses)
        rows.append({'type': 'near', 'hash': dist, 'path': f"{a['path']}  <->  {b['path']}", 'split': f"{a['split']} / {b['split']}", 'class': f"{a['class']} / {b['class']}", 'crosses_split': crosses})

    df = pd.DataFrame(rows)
    if len(df):
        df.to_csv(out_csv, index=False)

    print('=' * 70)
    print('DUPLICATE / NEAR-DUPLICATE LEAKAGE REPORT')
    print('=' * 70)
    print(f'Exact-duplicate groups found:          {len(exact_dupes)}')
    print(f'  ...of which cross a split boundary:  {cross_split_exact}')
    print(f'Near-duplicate pairs found:            {len(near_dupe_pairs)}')
    print(f'  ...of which cross a split boundary:  {cross_split_near}')
    print()
    print(f'Full row-level detail written to: {out_csv}')
    print('(this file is saved in /kaggle/working and will appear in the notebook\'s')
    print('Output tab / Data pane on the right, downloadable after the run finishes)')
    print('=' * 70)
    print('COPY EVERYTHING ABOVE THIS LINE (from the row of = signs at the top)')
    print('and send it back for the manuscript and response letter to be updated.')


train_dir = os.path.join(split_root, 'train')
val_dir = os.path.join(split_root, 'val')
test_dir = os.path.join(split_root, 'test')

records = []
records += collect_images(train_dir, 'train')
records += collect_images(val_dir, 'val')
records += collect_images(test_dir, 'test')
print(f'Found {len(records)} images across train/val/test.')

records = compute_hashes(records, hash_size=16)
exact_dupes, near_dupe_pairs = find_duplicates(records, near_dup_threshold=5)
summarize(exact_dupes, near_dupe_pairs, '/kaggle/working/duplicate_report.csv')

## 5. Getting the detailed CSV report

Unlike Colab, Kaggle doesn't need a special download call — anything written to `/kaggle/working` automatically shows up in the notebook's **Output** section (visible in the right-hand panel, or under the notebook's "Output" tab after the run completes / after you commit the notebook via **Save Version**). Click the file there to download `duplicate_report.csv` directly.

---
### Note: using your own exact split files instead of a fresh random split
If you have the exact list of which images went into train/val/test for the paper's actual experiments, that is a more faithful check than the fresh random split created in Section 3 above. To use them instead:
1. Upload your split files as a new Kaggle Dataset (Create → New Dataset → upload your text/CSV files), then Add it as a second input to this notebook
2. Replace Section 3's cell with code that symlinks/copies files into `/kaggle/working/split_dataset/train`, `/val`, `/test` according to your lists, instead of a fresh random shuffle
3. Re-run Section 4 as-is

If you don't have saved split files from the original run, the fresh-split version above is still meaningful: it tells you how many duplicate/near-duplicate images exist in the dataset at all, which is the core fact reviewers are asking about, even if the exact split-boundary-crossing count would differ slightly with a different random seed.

### Note on Kaggle session limits
Kaggle gives free notebooks a generous session length and no GPU is needed for this particular notebook (it only reads/hashes images, no model training), so this should comfortably finish in one sitting. If you do enable a GPU accidentally, that's fine too — it just won't be used here.